# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [1]:
import requests
from bs4 import BeautifulSoup
import time
import csv

In [12]:
import pandas as pd

In [5]:
def parse_f_atlas_simple():
    base_url = "https://f-atlas.ru"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}

    all_data = []
    page = 1

    while True:
        url = f"{base_url}/?page={page}/" if page > 1 else base_url

        try:
            response = requests.get(url, headers=headers, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')

            # Ищем все карточки (если есть общий класс)
            cards = soup.find_all(class_='catalog_item')  # Или 'item', 'card' и т.д.

            if not cards:
                # Если нет общего контейнера, ищем по заголовкам
                titles = soup.find_all(class_='catalog_item_title')
                for title in titles:
                    params = title.find_parent().find_all(class_='catalog_item_param')
                    all_data.append({
                        'name': title.get_text(strip=True),
                        'address': params[0].get_text(strip=True) if params else ""
                    })
            else:
                for card in cards:
                    title = card.find(class_='catalog_item_title')
                    params = card.find_all(class_='catalog_item_param')
                    if title and params:
                        all_data.append({
                            'name': title.get_text(strip=True),
                            'address': params[0].get_text(strip=True)
                        })

            # Проверяем наличие следующей страницы
            next_page = soup.find('a', string='Следующая')
            if not next_page:
                break

            page += 1
            time.sleep(1)

        except Exception as e:
            print(f"Ошибка: {e}")
            break

    return all_data

In [6]:
data=parse_f_atlas_simple()

Ошибка: HTTPSConnectionPool(host='f-atlas.ru', port=443): Read timed out. (read timeout=10)


In [8]:
len(data)

11360

In [9]:
data

[{'name': 'СИЗО-1 Кресты Колпино',
  'address': 'Санкт-Петербург и ЛО, г. Колпино ул. Колпинская д. 9 Сизо-1'},
 {'name': 'СИЗО-6 Москва Печатники',
  'address': 'г. Москва, ул. Шоссейная д. 92'},
 {'name': 'СИЗО-7 Капотня', 'address': 'г. Москва, ул. Верхние Поля д. 57'},
 {'name': 'СИЗО-4 Медведь', 'address': 'г. Москва, ул. Вилюйская д. 4'},
 {'name': 'СИЗО-1 Матросская тишина',
  'address': 'г. Москва, ул. Матросская Тишина д. 18, СИЗО-1'},
 {'name': 'СИЗО-2 Бутырка', 'address': 'г. Москва, ул. Новослободская д. 45'},
 {'name': 'СИЗО-1 Тверь', 'address': 'г. Тверь, ул. Вагжанова д. 141'},
 {'name': 'СИЗО-5 Водник',
  'address': 'г. Москва, ул. Выборгская д. 20 СИЗО-5'},
 {'name': 'СИЗО-2 Новокузнецк',
  'address': 'г. Новокузнецк, ул. Полосухина д. 3'},
 {'name': 'СИЗО-6 Горелово',
  'address': 'Санкт-Петербург и ЛО, Ломоносовский р-н поселение, ул. Заречная д. 22 СИЗО-6'},
 {'name': 'СИЗО-1 Екатеринбург',
  'address': 'г. Екатеринбург, ул. Репина д. 4, СИЗО-1'},
 {'name': 'СИЗО-1 

In [13]:
with open('institutions.csv', 'w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(file, fieldnames=['name', 'address'])
    writer.writeheader()
    writer.writerows(data)

In [14]:
df = pd.DataFrame(data)

In [15]:
df.head()

,name,address
0,СИЗО-1 Кресты Колпино,"Санкт-Петербург и ЛО, г. Колпино ул. Колпинска..."
1,СИЗО-6 Москва Печатники,"г. Москва, ул. Шоссейная д. 92"
2,СИЗО-7 Капотня,"г. Москва, ул. Верхние Поля д. 57"
3,СИЗО-4 Медведь,"г. Москва, ул. Вилюйская д. 4"
4,СИЗО-1 Матросская тишина,"г. Москва, ул. Матросская Тишина д. 18, СИЗО-1"


In [16]:
len(df)

11360